# BESS Peak-Shaving Load Forecasting Case Study

**Decision objective.** Forecast a German C&I site's 15-minute demand well enough to protect demand-charge peaks while preserving battery capacity for arbitrage. This case emphasizes data quality, temporal validation, interpretable baselines, and asymmetric operational risk.

**Executive finding.** The data is a 2025 15-minute series. Valid zero-load periods are structured shutdowns; 24 readings are missing or physically implausible and are flagged, with only isolated gaps interpolated. A weekly seasonal-naive forecast is a useful transparent reference, but a conservative 80th-percentile residual uplift reduces top-decile peak misses from 88.4% to 71.8% in the final 56-day holdout, at the cost of higher average error.

This notebook is deliberately compact and delegates reusable logic to `src/bess_forecasting.py`; `main.py` is the runnable entry point.

## 1. Pre-run checks, environment, and requirements
Run `python -m venv .venv`, activate it, and install with `python -m pip install -r requirements.txt`. The notebook expects pandas, numpy, matplotlib, and pytest.

## 2-5. Loading, timestamps, cleaning, resampling, and units
The source uses German date formatting, semicolon delimiters, and comma decimals. Timestamps are naive local time; 2025-03-30 contains a 02:00 to 03:00 jump consistent with a German DST transition. We standardize to a 15-minute power grid: kW observations are averaged when resampling, whereas kWh interval energy would be summed and converted to average kW. No timezone conversion is invented without site metadata.

Invalid values are defined as missing, negative, or above 200 kW, a conservative envelope relative to the cleaned 122.52 kW maximum. Valid zeros are retained because long zero runs match shutdown behavior. Only isolated invalid intervals are interpolated; longer outages remain missing and are excluded from model scoring.

## 6-7. Peak-shaving EDA and statistical summaries
The curated plots show the annual profile, a quantitatively selected representative week, an average weekday/weekend day, and the distribution of peak timestamps. The representative week is the complete Monday-Sunday week whose 15-minute profile is closest in mean absolute error to the median profile across all complete weeks. Daily maxima, peak persistence, weekday/weekend means, and load factor are printed below.

The final 56-day holdout remains untouched for scoring. Because the last December week contains holidays and shutdowns, the forecast chart uses a typical complete week from within the holdout rather than presenting that week as normal operations. This avoids visual cherry-picking while retaining holiday behavior in the aggregate metrics.

## 8. Features and leakage control
Calendar, lag, and rolling features are demonstrated below. Every lag uses only timestamps before the target. The production baseline uses a one-week seasonal lag, which is robust and interpretable; `add_calendar_features` provides the extension point for calendar/holiday features.

## 9-10. Baselines and explainability
The weekly seasonal-naive benchmark is `y_hat(t)=y(t-7 days)`. A persistence variant is the one-day lag. A linear regression with calendar and lag features is intentionally scoped to the module extension rather than selected as the default: with one year of one-site data, a stable seasonal reference is easier to audit and less likely to overfit process changes.

## 11-14. Temporal validation, asymmetric loss, and horizons
The final 56 calendar days are a chronological holdout; random k-fold is inappropriate because it leaks future operating regimes. We report MAE/RMSE plus a cost-weighted absolute error with under-forecast weight 2, under-forecast rate, and top-decile peak miss rate. The target-indexed seasonal benchmark is evaluated at 15-minute, 1-hour, and 4-hour labels; a production implementation should generate the full trajectory from each forecast origin.

## 15-16. Dispatch simulation and operating rules
A simple battery heuristic clips load above a threshold, constrained by power and state of charge. This is illustrative rather than a tariff settlement model: real savings require the demand-charge window, ratchet rules, efficiency, degradation, and market opportunity cost.

## 17-20. Entry point, utilities, tests, and reproducibility
`main.py` runs preparation, summary, and holdout evaluation. `tests/test_pipeline.py` covers locale parsing, cadence, zero preservation, artifact repair, asymmetric loss, and metric behavior. See `README.md` for commands and generated artifacts.

In [12]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path("c:/Users/DieVT/Downloads/Claude Machine Learning Projects/2026_BESS_PeakShavings")
sys.path.insert(0, str(ROOT))
import src.bess_forecasting as bess
bess = importlib.reload(bess)
load_and_prepare = bess.load_and_prepare
add_calendar_features = bess.add_calendar_features
asymmetric_metrics = bess.asymmetric_metrics
seasonal_forecast = bess.seasonal_forecast
evaluate_holdout = bess.evaluate_holdout
peak_summary = bess.peak_summary
plot_eda = bess.plot_eda
plot_forecast_comparison = bess.plot_forecast_comparison
select_representative_week = bess.select_representative_week

np.random.seed(42)
DATA = ROOT / "load_timeseries_2025_case_study.csv"
OUTPUT = ROOT / "outputs"
PLOT_DIR = OUTPUT / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

frame = add_calendar_features(load_and_prepare(DATA))
raw = pd.read_csv(DATA, sep=";", decimal=",")
print(f"Python {sys.version.split()[0]} | rows={len(frame):,} | range={frame.index.min()} to {frame.index.max()}")
print("Imports and reproducibility seed: OK")
print("\nRaw schema:")
print(raw.dtypes)
print(raw.head(3).to_string(index=False))
print("\nCadence:")
print(frame.index.to_series().diff().value_counts().head())
print("Missing timestamps inserted:", int((~frame.was_observed).sum()))
print("Duplicate timestamps in raw:", int(raw.iloc[:, 0].duplicated().sum()))
print("Artifact/interpolated/remaining missing:", int(frame.is_artifact.sum()), int(frame.was_interpolated.sum()), int(frame.load_kw.isna().sum()))

Python 3.9.5 | rows=35,040 | range=2025-01-01 00:00:00 to 2025-12-31 23:45:00
Imports and reproducibility seed: OK

Raw schema:
Timestamps     object
Load_kw       float64
dtype: object
      Timestamps  Load_kw
01.01.2025 00:00     2.38
01.01.2025 00:15     2.40
01.01.2025 00:30     5.12

Cadence:
timestamp
0 days 00:15:00    35039
Name: count, dtype: int64
Missing timestamps inserted: 157
Duplicate timestamps in raw: 0
Artifact/interpolated/remaining missing: 163 22 141


In [13]:
# Part 1: curated EDA
eligible = frame.loc[frame.is_scoring_eligible, "load_kw"]
summary = peak_summary(frame)
print(pd.Series(summary).to_string())
print("\nInvalid raw readings:")
print(raw.loc[pd.to_numeric(raw.Load_kw, errors="coerce").isna() | (pd.to_numeric(raw.Load_kw, errors="coerce") < 0) | (pd.to_numeric(raw.Load_kw, errors="coerce") > 200)].to_string(index=False))

representative_start, representative_end, profile_distance = select_representative_week(frame)
print(f"Representative EDA week: {representative_start:%d %b %Y} to {representative_end:%d %b %Y}")
print(f"Closest to median complete-week profile: {profile_distance:.2f} kW")
print("The forecast comparison uses a separate typical week inside the holdout, excluding the final holiday-heavy week.")

plot_eda(frame, PLOT_DIR)

# Peak persistence: consecutive intervals above the 90th percentile
is_peak = frame.load_kw.ge(eligible.quantile(.90)) & frame.is_scoring_eligible
run_id = is_peak.ne(is_peak.shift()).cumsum()
persistence = is_peak.groupby(run_id).sum()
print("Peak run duration (minutes), selected quantiles:")
print((persistence[persistence > 0] * 15).quantile([.5, .75, .9, 1]).to_string())

mean_kw                                  30.015735
p95_kw                                       83.36
max_kw                                      122.52
load_factor_vs_observed_max               0.244986
weekday_mean_kw                          40.411092
weekend_mean_kw                           3.936358
median_daily_peak_kw                         78.84
peak_timestamp                 2025-07-07 15:15:00
peak_hour                                    15.25

Invalid raw readings:
      Timestamps  Load_kw
13.01.2025 12:00      NaN
27.01.2025 01:00   999.99
16.02.2025 21:00    -4.00
04.03.2025 12:00      NaN
10.04.2025 00:00      NaN
10.04.2025 00:15      NaN
10.04.2025 00:30      NaN
06.05.2025 01:00      NaN
21.05.2025 16:00   -56.12
16.06.2025 17:00   999.99
07.07.2025 13:00      NaN
17.07.2025 23:00   546.80
18.08.2025 05:00      NaN
18.08.2025 05:15      NaN
28.09.2025 21:00   999.99
09.11.2025 13:00      NaN
09.11.2025 13:15      NaN
09.11.2025 13:30      NaN
09.11.2025 13:45      Na

In [ ]:
# Part 2: leakage-safe features and model comparison
features = frame[["load_kw", "lag_day_kw", "lag_week_kw"]].copy()
features["hour_sin"] = np.sin(2 * np.pi * frame.index.hour / 24)
features["hour_cos"] = np.cos(2 * np.pi * frame.index.hour / 24)
features["weekday"] = (frame.index.dayofweek < 5).astype(int)
model_data = features.dropna()
cutoff = frame.index.max() - pd.Timedelta(days=56) + pd.Timedelta(minutes=15)
train = model_data.index < cutoff
x_cols = ["lag_day_kw", "lag_week_kw", "hour_sin", "hour_cos", "weekday"]
coef = np.linalg.lstsq(np.c_[np.ones(train.sum()), model_data.loc[train, x_cols]], model_data.loc[train, "load_kw"], rcond=None)[0]
reg_pred = pd.Series(np.c_[np.ones(len(model_data)), model_data[x_cols]] @ coef, index=model_data.index)
reg_metrics = asymmetric_metrics(model_data.loc[model_data.index >= cutoff, "load_kw"], reg_pred.loc[reg_pred.index >= cutoff])
print("Regression holdout metrics:", reg_metrics)
print("Regression coefficients:")
print(pd.Series(coef, index=["intercept"] + x_cols).sort_values(key=abs, ascending=False).to_string())

metrics = evaluate_holdout(frame)
print(metrics[["horizon_15min", "model", "mae_kw", "rmse_kw", "underforecast_rate", "peak_underforecast_rate", "weighted_absolute_error_kw"]].to_string(index=False, float_format=lambda value: f"{value:.3f}"))
plot_forecast_comparison(frame, metrics, PLOT_DIR)
comparison_first_monday = cutoff + pd.Timedelta(days=(7 - cutoff.dayofweek) % 7)
comparison_start, comparison_end, _ = select_representative_week(frame, first_start=comparison_first_monday.strftime("%Y-%m-%d"), last_start="2025-12-15")
actual = frame.loc[comparison_start:comparison_end, "load_kw"]
naive = seasonal_forecast(frame, 1, 0.0, cutoff).loc[actual.index]
conservative = seasonal_forecast(frame, 1, .8, cutoff).loc[actual.index]
print(f"Forecast chart week: {comparison_start:%d %b %Y} to {comparison_end:%d %b %Y}")
print("Metrics cover the full chronological 56-day holdout; the saved forecast chart uses a typical non-holiday holdout week for readability.")

Regression holdout metrics: {'n': 5355.0, 'mae_kw': 15.045308803084762, 'rmse_kw': 20.122184889665082, 'underforecast_rate': 0.5073762838468721, 'underforecast_mae_kw': 14.02741249622787, 'peak_underforecast_rate': 0.9962756052141527, 'peak_underforecast_mae_kw': 30.119002598813424, 'weighted_absolute_error_kw': 22.162485227408034, 'peak_cutoff_kw': 80.24}
Regression coefficients:
weekday        17.101626
hour_sin       -2.396764
hour_cos       -1.881233
intercept      -1.803271
lag_week_kw     0.424715
lag_day_kw      0.232640
 horizon_15min                 model  mae_kw  rmse_kw  underforecast_rate  peak_underforecast_rate  weighted_absolute_error_kw
             1 weekly_seasonal_naive  15.998   24.772               0.451                    0.884                      22.216
             1      conservative_q80  21.642   29.026               0.187                    0.718                      24.693
             4 weekly_seasonal_naive  15.998   24.772               0.451            

In [4]:
# Optional calendar enrichment: German public holidays, known without using future load values.
try:
    import holidays
    de_holidays = holidays.country_holidays("DE", years=[2025])
    frame["is_public_holiday"] = pd.Index(frame.index.date).isin(de_holidays)
    print("German public-holiday intervals:", int(frame.is_public_holiday.sum()))
except ImportError:
    frame["is_public_holiday"] = False
    print("Install requirements.txt to enable German holiday flags.")

German public-holiday intervals: 864


In [5]:
# Part 3: simple dispatch sanity check and threshold rules
def dispatch(load_kw, forecast_kw, threshold_kw=80.0, battery_kwh=100.0, power_kw=50.0, efficiency=.92):
    load_kw = pd.Series(load_kw).astype(float)
    forecast_kw = pd.Series(forecast_kw, index=load_kw.index).astype(float)
    dispatch_kw = np.minimum(np.maximum(forecast_kw - threshold_kw, 0), power_kw)
    dispatch_kw = np.minimum(dispatch_kw, load_kw.clip(lower=0))
    energy_used = dispatch_kw * .25 / efficiency
    dispatch_kw = dispatch_kw.where(energy_used.cumsum() <= battery_kwh, 0.0)
    return pd.DataFrame({"load_kw": load_kw, "forecast_kw": forecast_kw, "discharge_kw": dispatch_kw, "net_load_kw": load_kw - dispatch_kw})

sim = dispatch(actual, conservative)
print("Dispatch sanity: max reduction (kW)=", round(sim.discharge_kw.max(), 2), "energy used (kWh)=", round((sim.discharge_kw * .25).sum(), 2))
print("Peak before/after:", round(sim.load_kw.max(), 2), round(sim.net_load_kw.max(), 2))
assert (sim.net_load_kw <= sim.load_kw + 1e-9).all()
assert (sim.discharge_kw * .25 / .92).sum() <= 100 + 1e-9

threshold_scenarios = pd.DataFrame({"threshold_kw": [70, 80, 90], "risk_posture": ["aggressive protection", "balanced", "arbitrage preserving"]})
threshold_scenarios["rule"] = threshold_scenarios.apply(lambda row: f"Weekdays 08:00-18:00: clip forecast above {row.threshold_kw} kW ({row.risk_posture})", axis=1)
print(threshold_scenarios.to_string(index=False))

from src.bess_forecasting import run_pipeline
run_pipeline(DATA, OUTPUT)
print("Artifacts:", [path.name for path in OUTPUT.glob("*.csv")])

Dispatch sanity: max reduction (kW)= 3.8 energy used (kWh)= 18.76
Peak before/after: 9.4 9.4
 threshold_kw          risk_posture                                                                    rule
           70 aggressive protection Weekdays 08:00-18:00: clip forecast above 70 kW (aggressive protection)
           80              balanced              Weekdays 08:00-18:00: clip forecast above 80 kW (balanced)
           90  arbitrage preserving  Weekdays 08:00-18:00: clip forecast above 90 kW (arbitrage preserving)
Artifacts: ['eda_summary.csv', 'forecast_metrics.csv', 'prepared_load.csv']
